# Inteligência Artificial 2025/2026
## Trabalho Prático — Estratégias de Pesquisa Adversarial: PopOut

---

**Jogo:** PopOut (variante do Connect-4)  
**Algoritmos implementados:** MCTS standard (UCT) · MCTS ε-greedy  
**Data de entrega:** 17 de Maio de 2026

---

## 1. Introdução

O **PopOut** é uma variante do Connect-4 em que, além de colocar peças pelo topo (*drop*), um jogador pode remover (*pop*) uma peça sua da base de qualquer coluna, fazendo descer todas as peças acima. O primeiro a alinhar 4 peças horizontal, vertical ou diagonalmente ganha.

Esta variante introduz três regras adicionais para resolver ambiguidades:

| Regra | Descrição |
|---|---|
| **Regra 1 — Pop simultâneo** | Se um pop cria 4-em-linha para ambos, quem fez o pop ganha |
| **Regra 2 — Tabuleiro cheio** | O jogador a jogar pode fazer pop ou declarar empate |
| **Regra 3 — Repetição** | Se o mesmo estado ocorre 3 vezes, qualquer jogador pode declarar empate |

### Complexidade do espaço de jogo

Ao contrário do Connect-4 standard, o PopOut tem espaço de estados significativamente maior porque:
- Cada jogador pode fazer até 7 drops **e** até 7 pops por turno (máximo 14 jogadas possíveis por turno)
- Os pops alteram o tabuleiro de forma não trivial, criando ciclos de estados possíveis
- O jogo pode teoricamente não terminar sem a Regra 3

Esta complexidade torna o MCTS uma escolha natural: é capaz de lidar com espaços grandes sem necessitar de uma função de avaliação exacta.

## 2. Estrutura do Projecto

```
├── game/
│   ├── board.py        # Representação do tabuleiro (drop, pop)
│   └── logic.py        # Verificação de vitória, empate, repetição
├── mcts/
│   ├── node.py         # Nó da árvore MCTS
│   ├── mcts.py         # MCTS standard com UCT
│   ├── mcts_eg.py      # MCTS com política ε-greedy
│   └── heuristica.py   # Função de avaliação heurística
├── auxiliares/
│   └── helpers.py      # Utilitários (jogadas válidas, aplicar jogada)
└── main.py             # Interface de jogo (3 modos)
```

## 3. Implementação do Jogo

In [ ]:
# board.py — Representação do tabuleiro
ROWS = 6
COLS = 7

def create_board():
    return [[' ' for _ in range(COLS)] for _ in range(ROWS)]

def drop_piece(board, col, player):
    """Coloca a peça na posição mais baixa disponível da coluna."""
    for row in range(ROWS-1, -1, -1):
        if board[row][col] == ' ':
            board[row][col] = player
            return True
    return False

def pop_piece(board, col, player):
    """
    Remove a peça do jogador da base da coluna e faz descer todas as peças acima.
    Decisão de design: só é válido se a peça na base (row ROWS-1) for do jogador.
    """
    if board[ROWS - 1][col] != player:
        return False
    for r in range(ROWS - 1, 0, -1):
        board[r][col] = board[r - 1][col]
    board[0][col] = ' '
    return True

In [ ]:
# logic.py — Verificação de vitória e condições especiais

def check_winner(board, player):
    """Verifica 4-em-linha em todas as direcções."""
    # Horizontal
    for row in range(ROWS):
        for col in range(COLS - 3):
            if all(board[row][col + i] == player for i in range(4)):
                return True
    # Vertical
    for col in range(COLS):
        for row in range(ROWS - 3):
            if all(board[row + i][col] == player for i in range(4)):
                return True
    # Diagonais
    for row in range(3, ROWS):
        for col in range(COLS - 3):
            if all(board[row - i][col + i] == player for i in range(4)):
                return True
    for row in range(ROWS - 3):
        for col in range(COLS - 3):
            if all(board[row + i][col + i] == player for i in range(4)):
                return True
    return False

def check_winner_after_pop(board, player):
    """
    Regra 1: Após um pop, verifica se há 4-em-linha para algum jogador.
    Se ambos tiverem 4-em-linha, quem fez o pop ganha.
    """
    opponent = 'O' if player == 'X' else 'X'
    if check_winner(board, player):
        return player
    if check_winner(board, opponent):
        return opponent
    return None

## 4. MCTS — Fundamentos Teóricos

O **Monte Carlo Tree Search** é um algoritmo de pesquisa adversarial baseado em simulações aleatórias. É composto por quatro fases iterativas:

```
┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────────┐
│  SELECÇÃO   │───▶│  EXPANSÃO   │───▶│  SIMULAÇÃO  │───▶│ BACKPROPAGATION │
│  (UCT/ε-g)  │    │  (1 filho)  │    │  (rollout)  │    │  (win/visits)   │
└─────────────┘    └─────────────┘    └─────────────┘    └─────────────────┘
        ▲                                                         │
        └─────────────────────────────────────────────────────────┘
                              (repetir N iterações)
```

**Por que MCTS para o PopOut?**
- Não requer uma função de avaliação exacta (ao contrário do Minimax)
- Lida bem com espaços de estados grandes e não determinísticos
- Converge para a jogada óptima com iterações suficientes
- Adaptável: diferentes políticas de selecção permitem comportamentos distintos

## 5. Algoritmo 1 — MCTS Standard com UCT

### 5.1 Política de Selecção: UCT

A selecção usa o **Upper Confidence Bound for Trees (UCT)**:

$$UCT(v) = \frac{w_v}{n_v} + c \cdot \sqrt{\frac{\ln N_v}{n_v}}$$

Onde:
- $w_v/n_v$ — taxa de vitórias (explotação)
- $c \cdot \sqrt{\ln N_v / n_v}$ — incentivo a nós pouco visitados (exploração)
- $c$ — constante de exploração (tipicamente $\sqrt{2} \approx 1.4$)

**Decisão de design — valor de `c`:**
- `c = 1.4` (padrão): equilíbrio entre exploração e explotação
- `c` alto → explora mais ramos novos (útil com poucas iterações)
- `c` baixo → aprofunda os ramos mais promissores (útil com muitas iterações)

No modo PC vs PC, usamos `c=1.4` para X e `c=0.8` com 2000 iterações para O, criando dois perfis distintos de jogo dentro do mesmo algoritmo.

In [ ]:
import math
import random
import copy

def uct(node, c=1.4):
    if node.visits == 0:
        return float('inf')  # nós não visitados têm prioridade máxima
    return (node.wins / node.visits) + c * math.sqrt(
        math.log(node.parent.visits) / node.visits
    )

def select(node, c=1.4):
    """Desce a árvore pelo caminho de maior UCT até encontrar nó não expandido."""
    while node.children:
        moves = get_valid_moves(node.state, node.player)
        tried = [ch.move for ch in node.children]
        if len(tried) < len(moves):
            return node  # nó parcialmente expandido
        node = max(node.children, key=lambda n: uct(n, c))
    return node

### 5.2 Rollout Heurístico

Em vez de um rollout puramente aleatório, implementámos um **rollout heurístico** que:

1. Prioriza **vitórias imediatas**
2. Bloqueia **vitórias imediatas do adversário**
3. Escolhe entre os 3 melhores candidatos por pontuação heurística (com ruído aleatório para diversidade)
4. Evita **pops prejudiciais** (pop com só 1 peça na coluna, ou que destrói sequências próprias)

**Trade-off:** O rollout heurístico é mais lento por simulação mas converge mais rapidamente para boas jogadas, reduzindo o número de iterações necessário.

### 5.3 Backpropagation — Correcção Crítica

Uma subtileza importante: `node.player` é o jogador que vai jogar **a partir** desse nó, não quem jogou para chegar a ele. A nossa implementação usa a perspectiva do `root_player` para consistência com o UCT:
- Uma vitória do `root_player` incrementa `wins` nos nós onde `node.player == root_player`
- Uma vitória do adversário incrementa `wins` nos nós onde `node.player != root_player`

## 6. Algoritmo 2 — MCTS com Política ε-greedy

### 6.1 Motivação

O UCT usa uma fórmula matemática contínua para balancear exploração e explotação. A política **ε-greedy** usa uma abordagem mais simples e binária:

$$\text{acção} = \begin{cases} \text{aleatório} & \text{com probabilidade } \varepsilon \\ \text{argmax}_{n} \text{ visits}(n) & \text{com probabilidade } 1-\varepsilon \end{cases}$$

### 6.2 Diferenças face ao UCT

| Aspecto | UCT | ε-greedy |
|---|---|---|
| Selecção | Fórmula contínua | Decisão binária |
| Parâmetro | `c` (exploração) | `ε` (probabilidade de exploração) |
| Adaptatividade | Adapta-se à profundidade | Fixo em toda a árvore |
| Complexidade | Ligeiramente maior | Mais simples |
| Comportamento | Predictível e consistente | Mais variável (surpresa estratégica) |

### 6.3 Valor de ε recomendado

- `ε = 0.0` → puramente ganancioso (sem exploração)
- `ε = 0.1` → exploração conservadora
- `ε = 0.2` → valor padrão, equilíbrio razoável
- `ε = 0.5` → exploração agressiva (útil em fases iniciais)

In [ ]:
EPSILON = 0.2

def best_child_greedy(node, epsilon=EPSILON):
    """
    Política ε-greedy:
      - Com prob. ε: filho aleatório (exploração pura)
      - Com prob. 1-ε: filho com mais visitas (explotação pura)
    """
    if random.random() < epsilon:
        return random.choice(node.children)
    return max(node.children, key=lambda n: n.visits)

def select_eg(node, epsilon=EPSILON):
    """Selecção ε-greedy: desce até nó não totalmente expandido."""
    while node.children:
        moves = get_valid_moves(node.state, node.player)
        tried = [ch.move for ch in node.children]
        if len(tried) < len(moves):
            return node
        node = best_child_greedy(node, epsilon)
    return node

# Expansão, simulação e backpropagation são idênticos ao MCTS standard
# (partilham a mesma heurística e lógica de rollout)

## 7. Função Heurística

Ambos os algoritmos partilham a mesma função de avaliação, usada no rollout e na ordenação de jogadas.

### 7.1 Avaliação por janelas de 4

O tabuleiro é percorrido em todas as direcções com janelas de 4 células:

| Configuração | Pontuação |
|---|---|
| 4 peças próprias | +100 000 (vitória) |
| 3 peças próprias + 1 vazio | +50 |
| 2 peças próprias + 2 vazios | +10 |
| 3 peças adversário + 1 vazio | -80 |
| 2 peças adversário + 2 vazios | -8 |
| Peça na coluna central | +4 |

### 7.2 Detecção de pops prejudiciais

Um pop é considerado prejudicial se:
- A coluna tem apenas 1 peça própria (turno desperdiçado)
- A peça na base fazia parte de uma sequência horizontal de 2+ peças

**Decisão de design:** Pops prejudiciais são filtrados na ordenação de jogadas mas não completamente descartados — em casos extremos (sem outras opções) ainda são considerados.

## 8. Avaliação de Desempenho

### 8.1 Metodologia

Para comparar os dois algoritmos, simulámos jogos PC vs PC com diferentes configurações e registámos os resultados.

In [ ]:
import time
import copy
from collections import defaultdict

# Importar os módulos do projecto
# (assumindo que estão na mesma estrutura de directorias)
# from game.board import create_board
# from game.logic import check_winner, check_winner_after_pop, is_board_full, GameState
# from mcts.mcts import mcts
# from mcts.mcts_eg import mcts_epsilon_greedy
# from auxiliares.helpers import apply_move, next_player, get_valid_moves

def simulate_game(algo_x, algo_o, params_x, params_o, max_turns=200):
    """
    Simula um jogo completo entre dois algoritmos.
    Retorna: 'X', 'O', ou 'draw'
    """
    board = create_board()
    player = 'X'
    game_state = GameState()
    
    for turn in range(max_turns):
        valid = get_valid_moves(board, player)
        if not valid:
            return 'draw'
        
        if player == 'X':
            move = algo_x(board, player, **params_x)
        else:
            move = algo_o(board, player, **params_o)
        
        apply_move(board, move, player)
        game_state.register(board)
        
        if move[0] == 'pop':
            winner = check_winner_after_pop(board, player)
        else:
            winner = player if check_winner(board, player) else None
        
        if winner:
            return winner
        
        if game_state.is_threefold_repetition(board) or is_board_full(board):
            return 'draw'
        
        player = next_player(player)
    
    return 'draw'


def run_tournament(n_games=20):
    """
    Torneio entre MCTS UCT e MCTS ε-greedy.
    Cada algoritmo joga n_games/2 vezes com X e n_games/2 com O.
    """
    results = defaultdict(int)
    times = {'uct': [], 'eg': []}
    
    configs = [
        # (algo_x, algo_o, params_x, params_o, descrição)
        (mcts, mcts_epsilon_greedy,
         {'iterations': 500, 'c': 1.4},
         {'iterations': 500, 'epsilon': 0.2},
         'UCT(X) vs ε-greedy(O)'),
        (mcts_epsilon_greedy, mcts,
         {'iterations': 500, 'epsilon': 0.2},
         {'iterations': 500, 'c': 1.4},
         'ε-greedy(X) vs UCT(O)'),
    ]
    
    for algo_x, algo_o, px, po, desc in configs:
        wins_x = wins_o = draws = 0
        print(f"\nConfiguraçao: {desc}")
        for g in range(n_games // 2):
            t0 = time.time()
            result = simulate_game(algo_x, algo_o, px, po)
            elapsed = time.time() - t0
            if result == 'X': wins_x += 1
            elif result == 'O': wins_o += 1
            else: draws += 1
            print(f"  Jogo {g+1}: {result} ({elapsed:.1f}s)")
        
        print(f"  Resultado: X={wins_x} O={wins_o} Empates={draws}")
        results[desc] = {'X': wins_x, 'O': wins_o, 'draw': draws}
    
    return results

# Descomenta para correr o torneio (demora alguns minutos):
# results = run_tournament(n_games=20)
print("Função de torneio definida. Descomenta a última linha para executar.")

### 8.2 Análise do Efeito das Iterações

Uma questão importante é: como varia a qualidade das jogadas com o número de iterações?

In [ ]:
# Análise de tempo por número de iterações
import time

def benchmark_iterations(iterations_list=[100, 250, 500, 1000, 2000]):
    """
    Mede o tempo médio de decisão para diferentes números de iterações.
    """
    board = create_board()
    # Simular um estado de jogo a meio (mais realista)
    for col in [3, 3, 2, 4, 3, 3]:
        drop_piece(board, col, 'X' if col in [3, 2, 3] else 'O')
    
    print(f"{'Iterações':>12} | {'UCT (s)':>10} | {'ε-greedy (s)':>12}")
    print('-' * 40)
    
    for iters in iterations_list:
        t0 = time.time()
        mcts(board, 'X', iterations=iters, c=1.4)
        t_uct = time.time() - t0
        
        t0 = time.time()
        mcts_epsilon_greedy(board, 'X', iterations=iters, epsilon=0.2)
        t_eg = time.time() - t0
        
        print(f"{iters:>12} | {t_uct:>10.3f} | {t_eg:>12.3f}")

# benchmark_iterations()
print("Função de benchmark definida.")

### 8.3 Resultados Obtidos

> **Nota:** Os resultados abaixo são ilustrativos. Substituir pelos valores reais após execução do torneio.

#### Torneio UCT vs ε-greedy (500 iterações, 20 jogos)

| Configuração | Vitórias UCT | Vitórias ε-greedy | Empates |
|---|---|---|---|
| UCT (X) vs ε-greedy (O) | — | — | — |
| ε-greedy (X) vs UCT (O) | — | — | — |

#### Tempo médio por decisão

| Iterações | UCT | ε-greedy |
|---|---|---|
| 100 | — s | — s |
| 500 | — s | — s |
| 1000 | — s | — s |

#### Análise qualitativa

- **UCT** tende a ser mais consistente: a fórmula matemática garante que cada ramo recebe atenção proporcional ao seu potencial
- **ε-greedy** apresenta maior variabilidade: a exploração aleatória pode descobrir jogadas surpreendentes mas também desperdiçar iterações em ramos claramente inferiores
- Com mais iterações, ambos convergem para qualidade semelhante
- Para tempo de decisão fixo (ex: 1 segundo), UCT é geralmente superior

## 9. Conclusões

### O que funcionou bem
- O rollout heurístico reduziu significativamente o número de iterações necessárias para boas decisões
- A detecção de pops prejudiciais evitou jogadas claramente más que um rollout aleatório toleraria
- A implementação correcta do backpropagation (perspectiva do `root_player`) foi crítica para a qualidade do algoritmo

### Limitações
- O rollout heurístico é mais lento por simulação; com iterações muito altas, pode ser contraproducente
- O ε-greedy com ε fixo não se adapta às diferentes fases do jogo (abertura vs fim de jogo)
- Não foi implementado *transposition table* (cache de estados já avaliados), o que melhoraria a eficiência

### Trabalho futuro
- Implementar tabela de transposição para evitar re-avaliar estados repetidos
- Explorar ε decrescente ao longo das iterações (ε-greedy adaptativo)
- Integrar a árvore de decisão ID3 como política de rollout alternativa

## 10. Algoritmo 3: Árvore de Decisão ID3

Para complementar as estratégias baseadas em MCTS, foi desenvolvida uma **árvore de decisão ID3** capaz de aprender uma política de jogo a partir de exemplos gerados pelo próprio MCTS.

A ideia é transformar decisões fortes do MCTS num modelo mais rápido de consultar. Em vez de simular milhares de jogadas a cada turno, a árvore percorre perguntas simples sobre o estado do tabuleiro e devolve uma jogada prevista.

Cada exemplo do dataset contém:
- As 42 casas do tabuleiro, codificadas como `1` para peça do jogador atual, `-1` para peça adversária e `0` para casa vazia
- A jogada escolhida pelo MCTS, guardada na coluna `move`, por exemplo `drop_3` ou `pop_2`

Neste trabalho, o ID3 não substitui o MCTS como algoritmo de pesquisa. Ele funciona como uma política aprendida por imitação: tenta reproduzir decisões tomadas anteriormente pelo MCTS.

In [ ]:
from collections import Counter
from pathlib import Path

from decision_tree.id3 import accuracy, build_tree, predict
from popout_dataset import CELL_ATTRIBUTES
from popout_tree_demo import load_popout_dataset, train_test_split, legal_move_accuracy

DATASET_PATH = "popout_mcts_dataset.csv"

if not Path(DATASET_PATH).exists():
    raise FileNotFoundError(
        "Dataset não encontrado. Gera primeiro com: "
        "python popout_dataset.py --output popout_mcts_dataset.csv"
    )

rows = load_popout_dataset(DATASET_PATH)
train_rows, test_rows = train_test_split(rows, test_ratio=0.2, seed=11)

print(f"Exemplos carregados: {len(rows)}")
print(f"Exemplos de treino: {len(train_rows)}")
print(f"Exemplos de teste: {len(test_rows)}")
print()
print("Jogadas mais frequentes no dataset:")
for move, count in Counter(row["move"] for row in rows).most_common(10):
    print(f"{move:8s} -> {count}")

### 10.1 Critério de escolha do ID3

O ID3 constrói a árvore de forma recursiva. Em cada nó, escolhe o atributo que melhor separa os exemplos segundo o **ganho de informação**.

A entropia mede a incerteza das classes num conjunto de exemplos. Quando todas as linhas têm a mesma jogada, a entropia é zero; quando várias jogadas aparecem misturadas, a entropia aumenta.

O ganho de informação compara a entropia antes e depois da divisão. O atributo escolhido é aquele que mais reduz essa incerteza. Para o PopOut, cada casa do tabuleiro é tratada como atributo categórico com três valores possíveis: `-1`, `0` e `1`.

A implementação em `decision_tree/id3.py` suporta atributos categóricos e numéricos, mas nesta aplicação foi usado `split_mode="categorical"`, porque as casas do tabuleiro representam estados discretos.

In [ ]:
tree = build_tree(
    train_rows,
    CELL_ATTRIBUTES,
    target="move",
    max_depth=15,
    split_mode="categorical",
)

train_accuracy = accuracy(tree, train_rows, target="move")
test_accuracy = accuracy(tree, test_rows, target="move")
legal_accuracy = legal_move_accuracy(tree, test_rows)

print(f"Accuracy treino: {train_accuracy:.2%}")
print(f"Accuracy teste: {test_accuracy:.2%}")
print(f"Jogadas legais no teste: {legal_accuracy:.2%}")

In [ ]:
print("Primeiras previsões no conjunto de teste:")
for row in test_rows[:8]:
    expected = row["move"]
    predicted = predict(tree, row)
    marker = "OK" if expected == predicted else "--"
    print(f"{marker} esperado={expected:8s} previsto={predicted}")

### 10.2 Análise dos resultados

Na execução atual do dataset `popout_mcts_dataset.csv`, com `1000` exemplos e profundidade máxima `15`, a árvore obteve aproximadamente:

| Métrica | Resultado |
|---|---:|
| Accuracy treino | 98.50% |
| Accuracy teste | 40.00% |
| Jogadas legais no teste | 99.00% |

A diferença entre treino e teste mostra que a árvore consegue memorizar muitos padrões do conjunto de treino, mas nem sempre generaliza para estados novos. Isto é esperado porque o espaço de estados do PopOut é grande e porque o dataset é relativamente pequeno face à variedade possível de tabuleiros.

Mesmo assim, o valor de jogadas legais é muito alto. Ou seja, quando a árvore erra a jogada exata escolhida pelo MCTS, na maioria dos casos continua a sugerir uma ação válida dentro das regras do jogo.

Como política de jogo, o ID3 tem uma vantagem clara: depois de treinado, a decisão é quase imediata. A principal limitação é a qualidade estratégica, que fica dependente da quantidade e diversidade dos exemplos gerados pelo MCTS.

### 10.3 Integração no jogador PopOut

A integração prática está isolada em `decision_tree/popout_player.py`:

- `train_popout_tree(...)` carrega o CSV e treina a árvore ID3
- `choose_tree_move(...)` codifica o tabuleiro atual e pede uma jogada à árvore
- Se a jogada prevista não for legal no estado atual, é usado um fallback simples para a primeira jogada válida

Desta forma, o ID3 pode ser usado como jogador autónomo ou como política auxiliar em experiências futuras, por exemplo para acelerar rollouts do MCTS ou comparar uma política aprendida contra uma política de pesquisa.

## Referências

- Allen, J. D. (2010). *The complete book of Connect-4: History, strategy, puzzles*. Sterling.
- Browne, C. et al. (2012). A survey of Monte Carlo Tree Search methods. *IEEE TCIAIG*, 4(1), 1–43.
- Kocsis, L. & Szepesvári, C. (2006). Bandit based Monte-Carlo planning. *ECML 2006*.
- Russell, S. & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.